# Leaving Money on the Table
## Notebook 01: Data Loading

Research question: (How do listings characteristics explain price differences in Berlin's Airbnb market and which listings fall below expected price?)
Data: Inside Airbnb, Berlin, snapshot (19 September 2026)

In [23]:
import pandas as pd

listings = pd.read_csv("../data/listings.csv.gz")

In [24]:
listings.shape

(12376, 90)

In [25]:
listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,1733333348320050763,https://www.airbnb.com/rooms/1733333348320050763,20260919134159,2026-09-20,NaN,Schöne Souterrain Wohnung unweit vom Ku'damm,Enjoy the simple life in this quiet and centra...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,1729156025960279337,...,5.0,5.0,4.0,04/Z/AZ/017613-25,NaN,1,1,0,0,1.00
1,811527740012470418,https://www.airbnb.com/rooms/811527740012470418,20260919134159,2026-09-20,NaN,Moderne Wohnung in Top-Lage,The whole group will enjoy easy access to ever...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,366280303,...,NaN,NaN,NaN,First name and Last name: Arthur Joselowitsch ...,NaN,1,1,0,0,NaN
2,638045311043310955,https://www.airbnb.com/rooms/638045311043310955,20260919134159,2026-09-20,NaN,Berlin - Neukölln - 2 bedroom flat,Well divided and spacious flat for families in...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,37827150,...,5.0,4.4,4.4,08/Z/RA/012501-23,NaN,1,1,0,0,0.12
3,20842113,https://www.airbnb.com/rooms/20842113,20260919134159,2026-09-20,previous scrape,Cosy Bright Room in Kreuzberg,"Cosy, bright room right in the heart of Kreuzb...",NaN,https://a0.muscache.com/pictures/793a5224-740a...,15510261,...,5.0,5.0,5.0,NaN,NaN,2,0,2,0,0.02
4,20833362,https://www.airbnb.com/rooms/20833362,20260919134159,2026-09-20,previous scrape,Quiet room in lively area.,3 minutes walk from U-bahn Karl-Marx-Str (U7)....,NaN,https://a0.muscache.com/pictures/2adf4b44-204d...,8677751,...,5.0,NaN,NaN,NaN,NaN,1,0,1,0,0.01


In [26]:
listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 12376 entries, 0 to 12375
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            12376 non-null  int64  
 1   listing_url                                   12376 non-null  str    
 2   scrape_id                                     12376 non-null  int64  
 3   last_scraped                                  12376 non-null  str    
 4   source                                        5815 non-null   str    
 5   name                                          12376 non-null  str    
 6   description                                   12305 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   12376 non-null  str    
 9   host_id                                       12376 non-null  int64  
 1

In [27]:
listings["price"].head(10)

0     $97.00
1     $69.60
2    $219.00
3        NaN
4        NaN
5        NaN
6     $65.67
7     $74.00
8    $172.26
9     $65.88
Name: price, dtype: str

In [28]:
listings["price"].isnull().sum()


np.int64(4219)

In [29]:
listings.groupby(listings["price"].isnull())["availability_365"].mean()

price
False    225.247763
True      28.945011
Name: availability_365, dtype: float64

### Handling missing prices

 Out of 12376 listings 4219 (about 34%) had no price.
When I compared their availability, listings without a price were available
  on average only 29 days per year, compared to 225 days for listings with
  a price. This suggests that listings with missing prices were either inactive or barely used.
Because price is the target variable, I decided to delete these rows,
  since filling them with an estimate would result into producing artificial values which would reult into biased model.
 After this step,  8157 listings remain for analysis.

In [30]:
df = listings.dropna(subset=["price"]).copy()
df.shape

(8157, 90)

In [31]:
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [32]:
df["price"].describe()

count     8157.000000
mean       167.564032
std        191.783955
min         12.860000
25%         81.140000
50%        129.500000
75%        203.000000
max      10025.000000
Name: price, dtype: float64

In [33]:
df["price"].quantile([0.90, 0.95, 0.99])

0.90    308.132
0.95    392.534
0.99    691.200
Name: price, dtype: float64

In [34]:
df.sort_values("price", ascending=False)[["name", "room_type", "accommodates", "price"]].head(15)

,name,room_type,accommodates,price
7928,Loft-Wohnung in Steglitz,Entire home/apt,7,10025.00
6031,"Event Space ""Headspace Harbour""",Entire home/apt,16,4457.70
573,New York style cosy loft in Berlin,Entire home/apt,2,3837.63
1027,250 qm penthouse with roof terrace for photoshoot,Entire home/apt,15,2956.00
6098,Lakeside Retreat near Berlin,Entire home/apt,16,2296.00
1961,Hausboot Shared Horizon,Entire home/apt,12,2270.40
6130,Brutalistischer Space zum Wohlfühlen,Entire home/apt,16,2262.00
7491,160qm2 Luxus Schwimmende Ferienwohnung+Sauna+K...,Entire home/apt,12,2100.00
11361,Luxury loft in Berlin Mitte,Entire home/apt,6,2033.33
2233,Gorki Penthouse 1,Entire home/apt,6,1995.00


In [35]:
cap = df["price"].quantile(0.99)
df = df[df["price"] <= cap].copy()
df.shape

(8075, 90)

In [36]:
df["price"].describe()

count    8075.000000
mean      156.653612
std       106.302451
min        12.860000
25%        80.770000
50%       127.920000
75%       200.000000
max       689.000000
Name: price, dtype: float64

### Handling extreme prices  

- After converting price to a number, the maximum price was €10,025,
  while 99% of listings cost less than €691.   
- Looking at the most expensive listings, many were not normal stays.
  For example, an event space at €4458 and a penthouse rented for photoshoots at €2956 and also loft for 2 guests at €3838 which is not actually realistic .
- I therefore removed listings above the 99th percentile (€691),
  because  they are not typical accommodation and would make the model less accurate for normal listings.
 - This removed 82 listings (about 1%), leaving 8075 listings.
 After this step, the standard deviation dropped from €191.78 to €106.30,
- while the median stayed almost the same (€129.50 → €127.92), which shows that a small number of listings changed the spread a lot but the typical price did not change so it means 82 listings were heavily distorting data.